# Image Preprocessing & Normalization Guide
### CIFAR-10 | PyTorch + torchvision | Kaggle Notebook

---

## What Is Image Preprocessing?

Raw images are just grids of pixel values — integers from 0 to 255 per channel. Before feeding them into a neural network, we need to:

- **Standardize** their size and format so every input looks the same
- **Normalize** pixel values so training is numerically stable
- **Augment** them so the model sees more variety and generalizes better
- **Enhance** them when needed (equalization, edge detection) to bring out features

Think of it like preparing ingredients before cooking — the better your prep, the better the result.

---

## Dataset

**Name:** CIFAR-10  
**Source:** [Kaggle](https://www.kaggle.com/competitions/cifar-10)  
**Description:** 60,000 color images (32×32 pixels, 3 RGB channels) across 10 classes — airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. Split into 50,000 training and 10,000 test images.

---

## Notebook Structure

| Step | Topic |
|------|-------|
| 1 | Setup & Imports |
| 2 | Load CIFAR-10 & First Look |
| 3 | Resizing & Cropping |
| 4 | Grayscale Conversion |
| 5 | Normalization (Min-Max & Mean/Std) |
| 6 | Histogram Equalization |
| 7 | Edge Detection |
| 8 | Noise Addition |
| 9 | Augmentation (Flip, Rotate, Zoom) |
| 10 | Full Training Pipeline |

---
## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset

import cv2
from PIL import Image, ImageFilter
from scipy.ndimage import gaussian_filter

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Plot style
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# CIFAR-10 class names
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'PyTorch version   : {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')
print(f'CUDA available    : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device            : {device}')

---
## 2. Load CIFAR-10 & First Look

We load CIFAR-10 with **no transforms** first — raw pixel values in `[0, 255]` as PIL Images. This gives us a clean baseline to compare every preprocessing step against.

The only transform we apply at load time is `ToTensor()` which converts PIL Images to PyTorch tensors with shape `(C, H, W)` and values in `[0.0, 1.0]`.

In [ ]:
# Load raw dataset — ToTensor only (no normalization yet)
raw_transform = T.ToTensor()  # converts PIL [0,255] → tensor [0.0, 1.0]

train_dataset = CIFAR10(
    root='/kaggle/input/cifar-10-python',
    train=True,
    download=False,
    transform=raw_transform
)
test_dataset = CIFAR10(
    root='/kaggle/input/cifar-10-python',
    train=False,
    download=False,
    transform=raw_transform
)

print(f'Train samples : {len(train_dataset):,}')
print(f'Test samples  : {len(test_dataset):,}')

# Inspect a single sample
img_tensor, label = train_dataset[0]
print(f'\nSingle image tensor shape : {img_tensor.shape}  (C, H, W)')
print(f'Pixel value range         : [{img_tensor.min():.3f}, {img_tensor.max():.3f}]')
print(f'Label                     : {label} ({CLASS_NAMES[label]})')

In [ ]:
# ── Helper: tensor to numpy image for plotting ────────────────────────────────
def to_numpy(tensor):
    """Convert (C, H, W) tensor in [0,1] to (H, W, C) numpy array for plt.imshow."""
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)

def show_grid(images, titles, suptitle, rows=2, cols=5, figsize=(14, 6)):
    """Plot a grid of images with titles."""
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:  # grayscale
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    plt.suptitle(suptitle, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Grab one image per class for visualization
class_samples = {}
for img, label in train_dataset:
    if label not in class_samples:
        class_samples[label] = img
    if len(class_samples) == 10:
        break

sample_imgs   = [to_numpy(class_samples[i]) for i in range(10)]
sample_titles = [CLASS_NAMES[i] for i in range(10)]

show_grid(sample_imgs, sample_titles,
          'CIFAR-10: One Sample per Class (Raw, 32x32)')

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
labels_all = [train_dataset[i][1] for i in range(len(train_dataset))]
counts = np.bincount(labels_all)

plt.figure(figsize=(10, 4))
plt.bar(CLASS_NAMES, counts, color='#4C72B0', edgecolor='white')
plt.title('CIFAR-10 Class Distribution (Training Set)', fontsize=13, fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=30)
for i, v in enumerate(counts):
    plt.text(i, v + 50, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pixel value distribution (raw) ───────────────────────────────────────────
# Sample 500 images to compute channel statistics
sample_tensors = torch.stack([train_dataset[i][0] for i in range(500)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
channel_names = ['Red', 'Green', 'Blue']
colors = ['#C44E52', '#55A868', '#4C72B0']

for i, (ax, name, color) in enumerate(zip(axes, channel_names, colors)):
    pixel_vals = sample_tensors[:, i, :, :].numpy().flatten()
    ax.hist(pixel_vals, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{name} Channel\nmean={pixel_vals.mean():.3f}, std={pixel_vals.std():.3f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Pixel Value [0, 1]')
    ax.set_ylabel('Frequency')

plt.suptitle('Raw Pixel Value Distribution per Channel (500 samples)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Resizing & Cropping

### Why Resize?

Neural networks require **fixed-size inputs**. If your dataset has images of varying dimensions, you must resize them to a consistent shape. Even with CIFAR-10 (already 32×32), resizing is a core technique to understand.

### Types of Resize Operations

| Operation | Description | Use Case |
|-----------|-------------|----------|
| `Resize(h, w)` | Stretch/shrink to exact size | When aspect ratio doesn't matter |
| `CenterCrop(size)` | Crop the center region | Remove borders, keep center content |
| `RandomCrop(size)` | Crop a random region | Data augmentation |
| `Resize + CenterCrop` | Resize then crop | Standard practice for ImageNet models |

**Caution:** Naive resizing can distort aspect ratios. Common practice: resize the shorter side, then crop.

In [ ]:
# Pick a fixed sample image for demonstrations
demo_tensor, demo_label = train_dataset[7]  # a cat
demo_pil = TF.to_pil_image(demo_tensor)

# Define resize/crop transforms
resize_ops = {
    'Original (32x32)'       : T.Compose([T.ToTensor()]),
    'Resize → 64x64'         : T.Compose([T.Resize((64, 64)),  T.ToTensor()]),
    'Resize → 16x16'         : T.Compose([T.Resize((16, 16)),  T.ToTensor()]),
    'CenterCrop 24x24'       : T.Compose([T.CenterCrop(24),    T.ToTensor()]),
    'Resize64 + CenterCrop32': T.Compose([T.Resize(64), T.CenterCrop(32), T.ToTensor()]),
    'Pad → 48x48'            : T.Compose([T.Pad(8), T.ToTensor()]),
}

imgs    = [to_numpy(op(demo_pil)) for op in resize_ops.values()]
titles  = list(resize_ops.keys())

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, img, title in zip(axes, imgs, titles):
    ax.imshow(img)
    ax.set_title(f'{title}\n{img.shape[0]}x{img.shape[1]}', fontsize=9)
    ax.axis('off')
plt.suptitle(f'Resize & Crop Operations — "{CLASS_NAMES[demo_label]}"',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Grayscale Conversion

### When Do You Need Grayscale?

Color images have 3 channels (RGB) — 3× more data than grayscale. Converting to grayscale:
- Reduces model input size (useful for lightweight models)
- Is required for some classical vision algorithms (edge detection, SIFT)
- Makes sense when **color is irrelevant** to the task (e.g., document OCR, texture classification)

**Grayscale formula:**
```
Gray = 0.299 × R + 0.587 × G + 0.114 × B
```
The weights reflect human perception — we're most sensitive to green light.

**Downside:** You lose color information that might be discriminative (e.g., traffic lights, fruit ripeness).

In [ ]:
grayscale_transform = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.ToTensor()
])

# Show RGB vs Grayscale for multiple classes
n_show = 5
rgb_imgs  = [to_numpy(class_samples[i]) for i in range(n_show)]
gray_imgs = [grayscale_transform(TF.to_pil_image(class_samples[i]))[0].numpy()
             for i in range(n_show)]

fig, axes = plt.subplots(2, n_show, figsize=(14, 5))
for i in range(n_show):
    axes[0, i].imshow(rgb_imgs[i])
    axes[0, i].set_title(CLASS_NAMES[i], fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(gray_imgs[i], cmap='gray')
    axes[1, i].set_title('grayscale', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('RGB', fontsize=11)
axes[1, 0].set_ylabel('Gray', fontsize=11)
plt.suptitle('RGB vs Grayscale Conversion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Channel-by-channel breakdown ─────────────────────────────────────────────
img_np = to_numpy(demo_tensor)  # (H, W, 3)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
titles_ch = ['Original', 'Red Channel', 'Green Channel', 'Blue Channel', 'Grayscale']

axes[0].imshow(img_np)
axes[1].imshow(img_np[:, :, 0], cmap='Reds')
axes[2].imshow(img_np[:, :, 1], cmap='Greens')
axes[3].imshow(img_np[:, :, 2], cmap='Blues')
axes[4].imshow(0.299*img_np[:,:,0] + 0.587*img_np[:,:,1] + 0.114*img_np[:,:,2], cmap='gray')

for ax, t in zip(axes, titles_ch):
    ax.set_title(t, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'Color Channel Decomposition — "{CLASS_NAMES[demo_label]}"',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Normalization

### Why Normalize?

Raw pixel values are in `[0, 255]` (or `[0.0, 1.0]` after `ToTensor()`). This causes two problems:

1. **Gradient instability**: Large input values → large gradients → exploding/vanishing gradients during backprop
2. **Unequal feature scales**: If R channel has mean 180 and B channel has mean 90, the network implicitly weights them differently at initialization

### Two Normalization Strategies

**Min-Max Normalization** — scales to `[0, 1]`:
```
x_norm = (x - x_min) / (x_max - x_min)
```

**Mean/Std Normalization (Standardization)** — centers to mean=0, std=1:
```
x_norm = (x - mean) / std
```
For CIFAR-10, the canonical per-channel statistics (computed over the full training set) are:
```
mean = [0.4914, 0.4822, 0.4465]
std  = [0.2470, 0.2435, 0.2616]
```
Using dataset-specific statistics ensures the input distribution matches what the pretrained model expects.

In [ ]:
# ── Compute CIFAR-10 mean and std from scratch ────────────────────────────────
# Load a larger sample to verify the canonical statistics

loader_tmp = DataLoader(
    Subset(train_dataset, range(10000)),
    batch_size=1000, shuffle=False
)

mean_accum = torch.zeros(3)
std_accum  = torch.zeros(3)
n_batches  = 0

for imgs_batch, _ in loader_tmp:
    # imgs_batch: (B, C, H, W)
    mean_accum += imgs_batch.mean(dim=[0, 2, 3])
    std_accum  += imgs_batch.std(dim=[0, 2, 3])
    n_batches  += 1

mean_computed = mean_accum / n_batches
std_computed  = std_accum  / n_batches

CIFAR_MEAN = mean_computed.tolist()
CIFAR_STD  = std_computed.tolist()

print(f'Computed mean (R, G, B): {[f"{v:.4f}" for v in CIFAR_MEAN]}')
print(f'Computed std  (R, G, B): {[f"{v:.4f}" for v in CIFAR_STD]}')
print(f'\nCanonical   mean: [0.4914, 0.4822, 0.4465]')
print(f'Canonical   std : [0.2470, 0.2435, 0.2616]')

In [ ]:
# ── Apply both normalization types and compare ────────────────────────────────

# Canonical CIFAR-10 stats
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

norm_meanstd = T.Normalize(mean=MEAN, std=STD)

raw_t    = demo_tensor                         # [0, 1] — after ToTensor
norm_t   = norm_meanstd(demo_tensor.clone())   # mean/std normalized

# Min-max is already done by ToTensor (0-255 → 0-1)
# Show the pixel value distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, tensor, title, color in zip(
    axes,
    [demo_tensor,
     (demo_tensor * 255).byte().float() / 255,
     norm_t],
    ['Raw [0, 255] via ToTensor → [0,1]',
     'Min-Max Normalized [0, 1]',
     'Mean/Std Normalized (~[-2, 2])'],
    ['#4C72B0', '#55A868', '#DD8452']
):
    vals = tensor.numpy().flatten()
    ax.hist(vals, bins=50, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'{title}\nmean={vals.mean():.3f}, std={vals.std():.3f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Pixel Value')
    ax.set_ylabel('Frequency')

plt.suptitle('Effect of Normalization on Pixel Value Distribution',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual comparison: before vs after normalization ─────────────────────────
# Note: normalized tensor has values outside [0,1] so we clip for display only

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for i in range(5):
    raw_img  = to_numpy(class_samples[i])
    norm_img = to_numpy(norm_meanstd(class_samples[i].clone()))

    axes[0, i].imshow(raw_img)
    axes[0, i].set_title(CLASS_NAMES[i], fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(norm_img)  # np.clip applied inside to_numpy
    axes[1, i].set_title('normalized', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Raw', fontsize=10)
axes[1, 0].set_ylabel('Mean/Std\nNormalized', fontsize=10)
plt.suptitle('Mean/Std Normalization — Visual Effect (clipped for display)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Histogram Equalization

### The Problem It Solves

Some images are **underexposed** (too dark) or **overexposed** (too bright). The pixel values cluster in a narrow range, leaving most of the dynamic range unused.

**Histogram equalization** redistributes pixel intensities so that they span the full range `[0, 255]` more uniformly. This dramatically improves contrast in dark or washed-out images.

**How it works:**
1. Compute the pixel intensity histogram
2. Compute its cumulative distribution function (CDF)
3. Use the CDF as a mapping function to remap each pixel

**CLAHE (Contrast Limited Adaptive Histogram Equalization)** is a better variant — it equalizes in small local tiles instead of globally, preventing over-amplification of noise.

We apply equalization to the **grayscale** version since it operates on single-channel intensity.

In [ ]:
def equalize_histogram(pil_img):
    """Apply global histogram equalization using PIL."""
    gray = pil_img.convert('L')
    from PIL import ImageOps
    equalized = ImageOps.equalize(gray)
    return equalized

def equalize_clahe(pil_img, clip_limit=2.0, tile_grid=(8, 8)):
    """Apply CLAHE using OpenCV."""
    gray = np.array(pil_img.convert('L'))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(gray)

# Demo on a few classes
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for i in range(5):
    pil_img = TF.to_pil_image(class_samples[i])
    gray_eq  = equalize_histogram(pil_img)
    clahe_eq = equalize_clahe(pil_img)

    axes[0, i].imshow(to_numpy(class_samples[i]))
    axes[0, i].set_title(CLASS_NAMES[i], fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(np.array(gray_eq), cmap='gray')
    axes[1, i].set_title('hist equalized', fontsize=9)
    axes[1, i].axis('off')

    axes[2, i].imshow(clahe_eq, cmap='gray')
    axes[2, i].set_title('CLAHE', fontsize=9)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Histogram\nEqualized', fontsize=10)
axes[2, 0].set_ylabel('CLAHE', fontsize=10)
plt.suptitle('Histogram Equalization vs CLAHE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Histogram before vs after equalization ───────────────────────────────────
pil_demo  = TF.to_pil_image(demo_tensor)
gray_orig = np.array(pil_demo.convert('L'))
gray_eq   = np.array(equalize_histogram(pil_demo))
gray_cl   = equalize_clahe(pil_demo)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax_img, ax_hist, img, title in zip(
    axes[0], axes[1],
    [gray_orig, gray_eq, gray_cl],
    ['Original (gray)', 'Hist Equalized', 'CLAHE']
):
    ax_img.imshow(img, cmap='gray')
    ax_img.set_title(title, fontsize=10, fontweight='bold')
    ax_img.axis('off')

    ax_hist.hist(img.flatten(), bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
    ax_hist.set_xlabel('Pixel Intensity')
    ax_hist.set_ylabel('Count')

plt.suptitle('Pixel Intensity Histograms: Before & After Equalization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Edge Detection

### What Are Edges?

Edges are boundaries in an image where pixel intensity **changes sharply**. They mark object outlines, texture boundaries, and structural features.

Edge detection is a key preprocessing step for:
- Classical computer vision (before deep learning dominated)
- Feature extraction for texture-based tasks
- Preprocessing for models that need structural emphasis over color

### Three Methods

| Method | Idea | Strength |
|--------|------|----------|
| **Sobel** | Gradient in X and Y directions separately | Directional edges |
| **Canny** | Multi-stage: gradient + non-max suppression + hysteresis | Clean, thin edges |
| **Laplacian** | Second derivative — detects rapid intensity changes | Fine details |

**Canny** is the gold standard for clean edge maps.

In [ ]:
def apply_edge_detection(pil_img):
    gray = np.array(pil_img.convert('L'))

    # Sobel
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel   = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel   = np.uint8(255 * sobel / sobel.max())

    # Canny
    canny = cv2.Canny(gray, threshold1=50, threshold2=150)

    # Laplacian
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = np.uint8(np.clip(np.abs(laplacian), 0, 255))

    return gray, sobel, canny, laplacian


fig, axes = plt.subplots(4, 5, figsize=(14, 10))
row_labels = ['Grayscale', 'Sobel', 'Canny', 'Laplacian']

for i in range(5):
    pil_img = TF.to_pil_image(class_samples[i])
    gray, sobel, canny, laplacian = apply_edge_detection(pil_img)

    for j, (img, cmap) in enumerate([
        (gray, 'gray'), (sobel, 'gray'), (canny, 'gray'), (laplacian, 'gray')
    ]):
        axes[j, i].imshow(img, cmap=cmap)
        axes[j, i].axis('off')
        if j == 0:
            axes[j, i].set_title(CLASS_NAMES[i], fontsize=9)

for j, label in enumerate(row_labels):
    axes[j, 0].set_ylabel(label, fontsize=10)

plt.suptitle('Edge Detection: Sobel vs Canny vs Laplacian',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Noise Addition

### Why Add Noise?

Adding noise is a form of **regularization through data augmentation**. Real-world images are never perfectly clean — cameras introduce sensor noise, images get compressed, lighting is imperfect.

Training on noisy images teaches the model to:
- Ignore irrelevant pixel-level variations
- Focus on structural and semantic features
- Generalize better to real-world deployment conditions

### Common Noise Types

| Noise Type | Description | Real-World Analogy |
|------------|-------------|--------------------|
| **Gaussian** | Random normal noise added to each pixel | Camera sensor noise |
| **Salt & Pepper** | Random pixels set to black or white | Transmission errors |
| **Speckle** | Multiplicative noise | Ultrasound / radar images |

In [ ]:
def add_gaussian_noise(tensor, mean=0.0, std=0.1):
    noise = torch.randn_like(tensor) * std + mean
    return torch.clamp(tensor + noise, 0.0, 1.0)

def add_salt_pepper_noise(tensor, prob=0.05):
    noisy = tensor.clone()
    salt   = torch.rand_like(tensor) < (prob / 2)
    pepper = torch.rand_like(tensor) < (prob / 2)
    noisy[salt]   = 1.0
    noisy[pepper] = 0.0
    return noisy

def add_speckle_noise(tensor, std=0.1):
    noise = torch.randn_like(tensor) * std
    return torch.clamp(tensor + tensor * noise, 0.0, 1.0)


fig, axes = plt.subplots(4, 5, figsize=(14, 10))
noise_fns  = [lambda t: t,
              lambda t: add_gaussian_noise(t, std=0.1),
              lambda t: add_salt_pepper_noise(t, prob=0.05),
              lambda t: add_speckle_noise(t, std=0.15)]
row_labels = ['Original', 'Gaussian\n(std=0.1)',
              'Salt & Pepper\n(p=0.05)', 'Speckle\n(std=0.15)']

for i in range(5):
    for j, fn in enumerate(noise_fns):
        noisy = fn(class_samples[i].clone())
        axes[j, i].imshow(to_numpy(noisy))
        axes[j, i].axis('off')
        if j == 0:
            axes[j, i].set_title(CLASS_NAMES[i], fontsize=9)

for j, label in enumerate(row_labels):
    axes[j, 0].set_ylabel(label, fontsize=9)

plt.suptitle('Noise Types: Gaussian vs Salt & Pepper vs Speckle',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gaussian noise at different intensities ───────────────────────────────────
std_levels = [0.0, 0.05, 0.10, 0.20, 0.40]
img_noisy  = [to_numpy(add_gaussian_noise(demo_tensor.clone(), std=s)) for s in std_levels]
titles_n   = [f'std = {s}' for s in std_levels]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, img, title in zip(axes, img_noisy, titles_n):
    ax.imshow(img)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
plt.suptitle('Gaussian Noise at Different Intensities',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Augmentation (Flip, Rotate, Zoom)

### What Is Data Augmentation?

Augmentation artificially expands your training set by **creating new versions of existing images** through geometric and color transformations.

**Why it works:**  
A cat is still a cat whether it's flipped horizontally, rotated 15°, or slightly zoomed in. By showing the model these variations during training, we teach it **invariance** — the ability to recognize objects regardless of orientation, scale, or position.

**Key rule:** Augmentation is applied **only during training**, never at inference. We don't want to evaluate on distorted images.

### torchvision Augmentations Covered

| Transform | Description |
|-----------|-------------|
| `RandomHorizontalFlip` | Mirror image left-right with probability p |
| `RandomVerticalFlip` | Mirror image top-bottom (less common) |
| `RandomRotation` | Rotate by a random angle in `(-degrees, +degrees)` |
| `RandomResizedCrop` | Zoom in to a random region then resize |
| `ColorJitter` | Random brightness, contrast, saturation, hue |
| `RandomGrayscale` | Convert to grayscale with probability p |
| `GaussianBlur` | Blur with random kernel size |

In [ ]:
# ── Show each augmentation individually ──────────────────────────────────────
aug_ops = {
    'Original'              : T.Compose([T.ToTensor()]),
    'HorizontalFlip'        : T.Compose([T.RandomHorizontalFlip(p=1.0), T.ToTensor()]),
    'VerticalFlip'          : T.Compose([T.RandomVerticalFlip(p=1.0),   T.ToTensor()]),
    'Rotate ±30°'           : T.Compose([T.RandomRotation(30),          T.ToTensor()]),
    'RandomResizedCrop'     : T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.ToTensor()]),
    'ColorJitter'           : T.Compose([T.ColorJitter(brightness=0.4, contrast=0.4,
                                                        saturation=0.4, hue=0.1), T.ToTensor()]),
    'GaussianBlur'          : T.Compose([T.GaussianBlur(kernel_size=3, sigma=(0.5, 2.0)), T.ToTensor()]),
    'RandomGrayscale'       : T.Compose([T.RandomGrayscale(p=1.0), T.ToTensor()]),
}

pil_demo = TF.to_pil_image(demo_tensor)

aug_imgs   = [to_numpy(op(pil_demo)) for op in aug_ops.values()]
aug_titles = list(aug_ops.keys())

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for ax, img, title in zip(axes, aug_imgs, aug_titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'Individual Augmentations — "{CLASS_NAMES[demo_label]}"',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Combined augmentation pipeline ───────────────────────────────────────────
# This is what you'd use as your training transform in practice

train_aug_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomResizedCrop(size=32, scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.RandomGrayscale(p=0.1),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std =[0.2470, 0.2435, 0.2616]),
])

# Show 10 different augmented versions of the same image
pil_demo = TF.to_pil_image(demo_tensor)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()
for i, ax in enumerate(axes):
    aug = train_aug_transform(pil_demo)
    ax.imshow(to_numpy(aug))
    ax.set_title(f'Aug #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'10 Random Augmentations of the Same Image — "{CLASS_NAMES[demo_label]}"\n'
             '(Combined Pipeline + Normalization)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Full Training Pipeline

### Putting It All Together

In a real training workflow, you define two separate transform pipelines:

- **Train transforms**: augmentation + normalization (more transformations = more generalization)
- **Test/Val transforms**: only resize + normalization (no augmentation — we evaluate on clean images)

This is the standard pattern used in almost every PyTorch image classification project.

```
Train image → Augment → Normalize → Model
Test  image →           Normalize → Model
```

In [ ]:
# ── Define train and test transforms ─────────────────────────────────────────

MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(32, padding=4),          # pad by 4 then random crop back to 32
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

print('Train transform:')
print(train_transform)
print('\nTest transform:')
print(test_transform)

In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────────────────

train_data = CIFAR10(
    root='/kaggle/input/cifar-10-python',
    train=True,
    download=False,
    transform=train_transform
)
test_data = CIFAR10(
    root='/kaggle/input/cifar-10-python',
    train=False,
    download=False,
    transform=test_transform
)

BATCH_SIZE = 128

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train batches : {len(train_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

# Verify a batch
imgs_batch, labels_batch = next(iter(train_loader))
print(f'\nBatch shape   : {imgs_batch.shape}  (B, C, H, W)')
print(f'Batch dtype   : {imgs_batch.dtype}')
print(f'Pixel range   : [{imgs_batch.min():.3f}, {imgs_batch.max():.3f}]')

In [ ]:
# ── Visualize a batch from the train loader ───────────────────────────────────
# Denormalize for display
mean_t = torch.tensor(MEAN).view(3, 1, 1)
std_t  = torch.tensor(STD).view(3, 1, 1)

def denormalize(tensor):
    return torch.clamp(tensor * std_t + mean_t, 0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i in range(8):
    raw_img  = to_numpy(denormalize(imgs_batch[i]))
    norm_img = to_numpy(imgs_batch[i])   # clipped to [0,1] by to_numpy

    axes[0, i].imshow(raw_img)
    axes[0, i].set_title(CLASS_NAMES[labels_batch[i].item()], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(norm_img)
    axes[1, i].set_title('normalized', fontsize=8)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Augmented', fontsize=10)
axes[1, 0].set_ylabel('Normalized', fontsize=10)
plt.suptitle('Training Batch: Augmented + Normalized (8 samples)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Pipeline Summary ──────────────────────────────────────────────────────────
print('============================================')
print('  IMAGE PREPROCESSING PIPELINE SUMMARY     ')
print('============================================')
print(f'  Dataset         : CIFAR-10')
print(f'  Image size      : 32x32 RGB')
print(f'  Train samples   : {len(train_data):,}')
print(f'  Test samples    : {len(test_data):,}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Normalization   : mean/std per channel')
print(f'    mean          : {MEAN}')
print(f'    std           : {STD}')
print(f'  Train augments  : HFlip, RandomCrop+pad,')
print(f'                    Rotation, ColorJitter')
print(f'  Test transforms : ToTensor + Normalize only')
print('============================================')